<a href="https://colab.research.google.com/github/LongLongoooo/Container_Code_Detection/blob/main/Container_Code_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls -lh "/content/drive/MyDrive/Container Shipping Number2-Forked on 8-19-2026.coco.zip"
!file "/content/drive/MyDrive/Container Shipping Number2-Forked on 8-19-2026.coco.zip"
# Nếu file hợp lệ, thử giải nén lại:
!unzip -q "/content/drive/MyDrive/Container Shipping Number2-Forked on 8-19-2026.coco.zip" -d /content/drive/MyDrive/Container_Code_Dataset

-rw------- 1 root root 104M Aug 19 06:15 '/content/drive/MyDrive/Container Shipping Number2-Forked on 8-19-2026.coco.zip'
/content/drive/MyDrive/Container Shipping Number2-Forked on 8-19-2026.coco.zip: Zip archive data, at least v2.0 to extract, compression method=store


In [ ]:
# Cài đặt thư viện YOLO
!pip install ultralytics

# Kiểm tra xem GPU đã nhận chưa
import torch
print("GPU is available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.5 MB/s eta 0:00:00
GPU is available: True


In [ ]:
import os
import glob

# ĐƯỜNG DẪN TỚI THƯ MỤC LABELS (thay đổi cho đúng với cấu trúc của bạn)
# Chạy lần lượt cho train, val, test
label_dir = '/content/dataset/valid/labels' # Nhớ đổi đường dẫn

# Tìm tất cả file .txt
txt_files = glob.glob(os.path.join(label_dir, '*.txt'))

for file_path in txt_files:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) > 0:
            # Lấy class ID hiện tại (ép kiểu về số nguyên)
            current_id = int(parts[0])

            # Trừ đi 1
            new_id = current_id - 1

            # Ráp lại thành dòng mới
            new_line = f"{new_id} {' '.join(parts[1:])}\n"
            new_lines.append(new_line)

    # Ghi đè lại file
    with open(file_path, 'w') as f:
        f.writelines(new_lines)

print(f"Đã cập nhật xong ID cho thư mục: {label_dir}")

Đã cập nhật xong ID cho thư mục: /content/dataset/valid/labels


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import glob

# Định nghĩa class name
class_names = ['container_code', 'iso_type']

# ĐƯỜNG DẪN TỚI THƯ MỤC ẢNH VÀ LABEL TRAIN (nhớ sửa cho đúng)
img_dir = '/content/dataset/train/images'
label_dir = '/content/dataset/train/labels'

# Lấy thử 3 ảnh đầu tiên
img_files = glob.glob(os.path.join(img_dir, '*.jpg'))[:3]

plt.figure(figsize=(15, 10))

for i, img_path in enumerate(img_files):
    # Đọc ảnh
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    # Tìm file label tương ứng
    txt_name = os.path.basename(img_path).replace('.jpg', '.txt')
    txt_path = os.path.join(label_dir, txt_name)

    # Đọc tọa độ và vẽ box
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center, y_center, width, height = map(float, parts[1:5])

                    # Chuyển đổi tọa độ chuẩn hóa về pixel
                    x_min = int((x_center - width/2) * w)
                    y_min = int((y_center - height/2) * h)
                    x_max = int((x_center + width/2) * w)
                    y_max = int((y_center + height/2) * h)

                    # Vẽ hộp chữ nhật (đỏ cho container_code, xanh cho iso_type)
                    color = (255, 0, 0) if class_id == 0 else (0, 0, 255)
                    cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, 3)

                    # Thêm nhãn
                    label = class_names[class_id] if class_id < len(class_names) else f"Unknown ID: {class_id}"
                    cv2.putText(img, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1500x1000 with 0 Axes>

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
print(model.ckpt["train_args"])


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'task': 'detect', 'mode': 'train', 'model': 'yolo11n.yaml', 'data': '/usr/src/ultralytics/ultralytics/cfg/datasets/coco.yaml', 'epochs': 600, 'time': None, 'patience': 100, 'batch': 128, 'imgsz': 640, 'save': True, 'save_period': -1, 'cache': 'disk', 'device': 0, 'workers': 8, 'project': 'exp10-new', 'name': 'yolov8n-c3k2-6', 'exist_ok': False, 'pretrained': True, 'optimizer': 'auto', 'verbose': True, 'seed': 0, 'deterministic': True, 'single_cls': False, 'rect': False, 'cos_lr': False, 'close_mosaic': 10, 'resume': False, 'amp': True, 'fraction': 1.0, 'profile': False, 'freeze': None, 'multi_scale': False, 'overlap_mask': True, 'mask_ratio': 4, 'dropout': 0.0, 'val': True, 'spli

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình
model = YOLO('yolo11n.pt')

# Tiến hành huấn luyện với cấu hình đã chuẩn hóa
results = model.train(
    data='/content/drive/MyDrive/Container_Code_Dataset/data.yaml',
    epochs=1,                  # Giảm xuống 50 để chạy thử trên CPU
    imgsz=640,
    batch=8,                    # Batch nhỏ để phù hợp CPU/RAM
    device='cpu',
    name='container_yolo_final',
    patience=20
)

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Container_Code_Dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=co

ValueError: [34m[1mtrain: [0mNo labels found in /content/drive/MyDrive/Container_Code_Dataset/train.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.